In [ ]:
import spcm
from spcm import units

with spcm.Card(card_type=spcm.SPCM_TYPE_AO, verbose=True) as card:
    print(f"Serial number:    {card.sn()}")
    print(f"Function type:    {card.function_type()}")
    print(f"Max sample value: {card.max_sample_value()}")

    clock = spcm.Clock(card)
    max_rate = clock.sample_rate(max=True, return_unit=units.MHz)
    print(f"Max sample rate:  {max_rate}")

print("Driver check OK -- card opened, queried, and closed without errors.")


In [3]:
import pyvisa

rm = pyvisa.ResourceManager()
print(rm.list_resources())


('ASRL1::INSTR', 'ASRL2::INSTR', 'ASRL3::INSTR', 'ASRL4::INSTR', 'ASRL5::INSTR', 'USB0::0xF4EC::0x1301::SVA1XA1Q800517::INSTR')


In [4]:
SVA_RESOURCE = "USB0::0xF4EC::0x1301::SVA1XA1Q800517::INSTR"

sva = rm.open_resource(SVA_RESOURCE)
sva.timeout = 5000
sva.read_termination = "\n"
sva.write_termination = "\n"

idn = sva.query("*IDN?").strip()
print(f"Connected to: {idn}")
assert "SVA1015X" in idn, f"Unexpected instrument reply: {idn!r}"


Connected to: Siglent Technologies,SVA1015X,SVA1XA1Q800517,3.2.2.6.0R10


In [7]:
def set_span(center_hz, span_hz):
    """Center the analyzer's sweep on center_hz with the given span (Hz)."""
    sva.write(f":SENSe:FREQuency:CENTer {center_hz}")
    sva.write(f":SENSe:FREQuency:SPAN {span_hz}")


def read_peak():
    """Move marker 1 to the highest peak in the current sweep and read it back
    as (freq_hz, amplitude_dbm).
    """
    sva.write(":CALCulate:MARKer1:STATe ON")
    sva.write(":CALCulate:MARKer1:MAXimum")
    freq_hz = float(sva.query(":CALCulate:MARKer1:X?"))
    amp_dbm = float(sva.query(":CALCulate:MARKer1:Y?"))
    return freq_hz, amp_dbm


set_span(center_hz=80e6, span_hz=80e6)
print("Spectrum analyzer ready.")


Spectrum analyzer ready.


In [ ]:
import spcm

from awg_controller.scripts.atommover_controller import HardwareConfig
from awg_controller.src.awg_control import AODSettings, AWGBatch, RFRamp
from awg_controller.src.scapp import ScappFeeder, ScappFeederConfig

hw = HardwareConfig(card_path="/dev/spcm0", max_amplitude_v=1.0)
aod = AODSettings(f_min_v=60e6, f_max_v=100e6, f_min_h=60e6, f_max_h=100e6, grid_rows=1, grid_cols=1)


def hold_at(freq_hz):
    """Single-tone holding batch at freq_hz, full 40% per-tone budget (only tone on this channel)."""
    return AWGBatch(
        ramps=[RFRamp(channel=0, core=0, f_start=freq_hz, f_end=freq_hz, amplitude_pct=40.0, tone_index=0)],
        total_duration_s=0.0,
    )


if "feeder" in globals() and feeder is not None:
    feeder.stop()
    feeder = None
if "card" in globals() and card is not None:
    card.close()
    card = None

card = spcm.Card(hw.card_path)
feeder = ScappFeeder(card, hw, aod, ScappFeederConfig(ramp_shape="linear"))
feeder.start(hold_at(aod.f_min_v))
print(f"Feeder started -- sample_rate={feeder.sample_rate_hz / 1e6:.1f} MHz, "
      f"holding at {aod.f_min_v / 1e6:.0f} MHz.")


In [ ]:
OBSERVABLE_RAMP_S = 3.0

ramp_batch = AWGBatch(
    ramps=[RFRamp(channel=0, core=0, f_start=aod.f_min_v, f_end=aod.f_max_v, amplitude_pct=40.0, tone_index=0)],
    total_duration_s=OBSERVABLE_RAMP_S,
)
print(f"Ramping {aod.f_min_v / 1e6:.0f} -> {aod.f_max_v / 1e6:.0f} MHz over "
      f"{OBSERVABLE_RAMP_S:.0f} s -- watch the spectrum analyzer/scope now.")
feeder.submit_batch(ramp_batch)
feeder.submit_holding(hold_at(aod.f_max_v))
print(f"Ramp complete -- now holding at {aod.f_max_v / 1e6:.0f} MHz. "
      f"dropped_transition_count={feeder.dropped_transition_count}")

if "sva" in globals() and sva is not None:
    freq_hz, amp_dbm = read_peak()
    print(f"SVA1015X peak: {freq_hz / 1e6:.3f} MHz @ {amp_dbm:.1f} dBm "
          f"(expected ~{aod.f_max_v / 1e6:.0f} MHz)")


In [ ]:
from atommovr.utils.timing import MIN_MOVE_DURATION_S

EXPERIMENT_RAMP_S = MIN_MOVE_DURATION_S

fast_batch = AWGBatch(
    ramps=[RFRamp(channel=0, core=0, f_start=aod.f_max_v, f_end=aod.f_min_v, amplitude_pct=40.0, tone_index=0)],
    total_duration_s=EXPERIMENT_RAMP_S,
)

dropped_before = feeder.dropped_transition_count
feeder.submit_batch(fast_batch)
feeder.submit_holding(hold_at(aod.f_min_v))
dropped_after = feeder.dropped_transition_count

print(f"Experiment-timescale ramp: {EXPERIMENT_RAMP_S * 1e6:.1f} us, "
      f"{aod.f_max_v / 1e6:.0f} -> {aod.f_min_v / 1e6:.0f} MHz")
print(f"dropped_transition_count: {dropped_before} -> {dropped_after}")
if dropped_after > dropped_before:
    print("WARNING: this transition never reached the DAC -- see ScappFeeder "
          "docstring; consider raising notify_samples.")
else:
    print("OK -- transition was rendered within the DMA chunk timing.")

assert feeder.last_error is None, feeder.last_error

if "sva" in globals() and sva is not None:
    freq_hz, amp_dbm = read_peak()
    print(f"SVA1015X peak: {freq_hz / 1e6:.3f} MHz @ {amp_dbm:.1f} dBm "
          f"(expected ~{aod.f_min_v / 1e6:.0f} MHz)")


In [ ]:
if "feeder" in globals() and feeder is not None:
    feeder.stop()
    print("Feeder stopped.")
    feeder = None

if "card" in globals() and card is not None:
    card.close()
    print("Card closed.")
    card = None

if "sva" in globals() and sva is not None:
    sva.close()
    print("Spectrum analyzer connection closed.")
    sva = None
